# Descrição do notebook

Este notebook realiza a preparação, harmonização, limpeza, normalização e consolidação de dados meteorológicos do estado do Rio de Janeiro, referentes aos anos de 2018 a 2024.

## 1. Documentação do fluxo

O notebook inicia descrevendo as principais etapas do processamento:

- Criação de uma sessão Spark;
- Leitura dos arquivos CSV;
- Comparação dos cabeçalhos;
- Padronização das colunas;
- Harmonização dos arquivos;
- Limpeza e normalização dos dados;
- União dos datasets;
- Exportação dos resultados processados.

## 2. Importação das bibliotecas

São utilizadas bibliotecas para manipulação de arquivos, registro de mensagens, processamento distribuído e análise tabular:

- `os` e `pathlib` para manipulação de caminhos;
- `logging` para registrar informações e erros;
- `pyspark` para processamento distribuído;
- `pandas` para leitura, transformação e concatenação dos arquivos CSV.

Também são importadas funções do Spark para conversão, substituição e tratamento de colunas.

## 3. Configuração do logging

O sistema de logging é configurado no nível `INFO`. Dessa forma, o notebook registra mensagens sobre:

- Arquivos carregados;
- Arquivos processados;
- Arquivos salvos;
- Erros encontrados durante a execução.



## 4. Definição dos arquivos de entrada

São definidos os caminhos dos arquivos CSV correspondentes aos anos de 2018 a 2024, armazenados no diretório:

```text
data/staged/
```

Cada arquivo representa um conjunto anual de dados meteorológicos do Rio de Janeiro.

## 5. Criação da SparkSession

O notebook verifica se já existe uma variável `spark`.

- Caso exista, ela é reutilizada;
- Caso contrário, uma nova `SparkSession` é criada com o nome `WeatherDataCleaning`.

Essa sessão permite utilizar o Apache Spark para leitura, transformação e gravação dos dados.

## 6. Leitura dos arquivos com Spark

A função `read_dataset_with_spark` lê os arquivos CSV utilizando:

- A primeira linha como cabeçalho;
- Inferência automática dos tipos das colunas;
- Registro de sucesso ou erro no processamento.

Em caso de falha, a função registra a exceção e retorna `None`.

## 7. Verificação e comparação dos cabeçalhos

A função `check_and_compare_headers` utiliza o pandas para ler apenas os cabeçalhos dos arquivos.

O procedimento:

1. Lê os nomes das colunas de cada arquivo;
2. Armazena os cabeçalhos em um dicionário;
3. Compara os cabeçalhos entre os anos;
4. Informa se todos possuem a mesma estrutura.

Essa etapa identifica diferenças de nomenclatura ou quantidade de colunas antes da harmonização.

## 8. Definição do esquema padrão

É criada a lista `colunas_padrao`, contendo os nomes esperados para todas as colunas dos datasets.

As colunas abrangem informações como:

- Data e hora da medição;
- Precipitação;
- Pressão atmosférica;
- Radiação solar;
- Temperatura;
- Umidade relativa;
- Direção e velocidade do vento;
- Localização da estação;
- Código WMO;
- Latitude, longitude e altitude;
- Data de fundação da estação.

## 9. Harmonização dos cabeçalhos

A função `substituir_cabecalhos_para_padrao` aplica o esquema padrão aos arquivos originais.

Durante essa etapa:

- Os arquivos são lidos com pandas;
- Colunas ausentes são adicionadas com valores nulos;
- Os nomes das colunas são substituídos pelos nomes padronizados;
- As colunas são reorganizadas na ordem definida;
- Novos arquivos são salvos com o prefixo `padrao_`.

Os arquivos harmonizados são armazenados no diretório `data/staged`.

## 10. Validação dos arquivos harmonizados

Após a substituição dos cabeçalhos, os novos arquivos são novamente verificados.

O notebook confirma se todos os arquivos harmonizados possuem:

- A mesma quantidade de colunas;
- Os mesmos nomes;
- A mesma ordem de colunas.

## 11. Limpeza e normalização com Spark

A função `limpar_e_normalizar_spark` realiza o tratamento dos dados utilizando PySpark.

### Remoção de valores nulos

As linhas que possuem valores nulos nas colunas definidas em `colunas_padrao` são removidas.

### Conversão de valores numéricos

As colunas meteorológicas e geográficas são tratadas como numéricas. Para isso:

1. As vírgulas decimais são substituídas por pontos;
2. Os valores são convertidos para o tipo `double`.

São processadas colunas de precipitação, pressão, radiação, temperatura, umidade, vento e coordenadas geográficas.

## 12. Normalização de datas e horários

A função `normalizar_colunas_tempo` padroniza os campos temporais.

### Coluna `data`

A coluna é convertida para o tipo `DATE`, considerando formatos como:

- `yyyy/MM/dd`;
- `dd/MM/yy`.

### Coluna `hora_utc`

A expressão ` UTC` é removida. Além disso, horários no formato `HHMM`, como `1430`, são convertidos para `14:30`.

### Coluna `data_de_fundacao`

Essa coluna também é convertida para o tipo `DATE`, utilizando os mesmos formatos possíveis da coluna `data`.

## 13. Processamento individual de um dataset

O arquivo de 2018 é utilizado como exemplo para demonstrar o fluxo completo:

1. Leitura com Spark;
2. Limpeza dos valores;
3. Conversão das colunas numéricas;
4. Normalização das datas e horários;
5. Exibição das primeiras linhas do resultado.

## 14. União dos datasets

A função `processar_e_salvar_todos_datasets` executa o processamento para todos os anos.

Para cada arquivo:

1. O dataset é lido com Spark;
2. Os valores são limpos e normalizados;
3. As colunas temporais são convertidas;
4. O DataFrame processado é armazenado em uma lista.

Em seguida, os DataFrames são unidos utilizando `unionByName`, preservando a correspondência entre os nomes das colunas.

## 15. Exportação com Spark

O DataFrame final processado pelo Spark é salvo no diretório de dados processados:

```text
/Volumes/workspace/mvp/processed/
```

O arquivo é gravado no formato CSV, com cabeçalho e modo `overwrite`.

Como o Spark normalmente grava os dados em um diretório contendo arquivos particionados, o caminho informado representa uma pasta de saída.

## 16. Consolidação alternativa com pandas

Além do processamento com Spark, os arquivos harmonizados são lidos novamente com pandas.

Os DataFrames anuais são concatenados em um único DataFrame chamado `df_concat` e salvos como:

```text
weather_data_cleaned_pandas.csv
```

Essa etapa fornece uma alternativa de consolidação utilizando processamento local com pandas.

## 17. Substituição de vírgulas decimais

É criada a função `substituir_virgula_por_ponto_em_csv`, que percorre o arquivo CSV linha a linha e substitui padrões numéricos com vírgula decimal por ponto.

Por exemplo:

```text
3,1
```

é convertido para:

```text
3.1
```

O resultado é salvo como:

```text
weather_data_cleaned_final.csv
```

## 18. Conversão das colunas numéricas no pandas

As colunas numéricas são limpas para remover aspas simples e duplas.

Depois, são convertidas para valores numéricos utilizando `pd.to_numeric`. Valores que não podem ser convertidos são transformados em `NaN`.

Também é realizada a substituição de vírgulas por pontos antes da conversão.

## 19. Conversão das colunas de data

As colunas `data` e `data_de_fundacao` são convertidas para o tipo datetime com `pd.to_datetime`.

Datas inválidas ou incompatíveis são convertidas para valores nulos (`NaT`).

## 20. Salvamento da versão final

Após os tratamentos realizados com pandas, o DataFrame final é salvo como:

```text
weather_data_cleaned_final_v2.csv
```

Esse arquivo representa a versão consolidada, com colunas numéricas convertidas e campos de data normalizados.

## Resultado final

Ao final do notebook, os dados meteorológicos de 2018 a 2024 estão:

- Com cabeçalhos padronizados;
- Integrados em uma estrutura única;
- Com valores numéricos normalizados;
- Com datas convertidas;
- Com horários tratados;
- Prontos para análises estatísticas, visualizações e aplicações posteriores.

In [0]:
import os
import logging
from pathlib import Path
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, when
from pyspark.sql.types import DoubleType
import pandas as pd

In [0]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [0]:
dataset_2018_rj = Path("data/staged/2018.csv")
dataset_2019_rj = Path("data/staged/2019.csv")
dataset_2020_rj = Path("data/staged/2020.csv")
dataset_2021_rj = Path("data/staged/2021.csv")
dataset_2022_rj = Path("data/staged/2022.csv")
dataset_2023_rj = Path("data/staged/2023.csv")
dataset_2024_rj = Path("data/staged/2024.csv")

In [0]:
try:
	spark
	logger.info("SparkSession já existe (Databricks ou ambiente similar).")
except NameError:
	spark = SparkSession.builder.appName("WeatherDataCleaning").getOrCreate()
	logger.info("SparkSession criada manualmente.")

INFO:__main__:SparkSession já existe (Databricks ou ambiente similar).


In [0]:
def read_dataset_with_spark(dataset_path):
    """
    Lê um arquivo CSV usando Spark e retorna um DataFrame do Spark.
    """
    try:
        df = spark.read.option("header", True).option("inferSchema", True).csv(str(dataset_path))
        logger.info(f"Dataset carregado com sucesso: {dataset_path}")
        return df
    except Exception as e:
        logger.exception(f"Erro ao ler o dataset: {dataset_path}")
        return None

In [0]:
def check_and_compare_headers(*dataset_paths):
    """
    Lê os cabeçalhos de cada dataset e compara se todos são iguais.
    Retorna um dicionário com os cabeçalhos e um booleano indicando se são iguais.
    """
    headers = {}
    for path in dataset_paths:
        try:
            # Lê apenas o cabeçalho usando pandas
            df = pd.read_csv(path, nrows=0)
            headers[str(path)] = list(df.columns)
        except Exception as e:
            logger.exception(f"Erro ao ler cabeçalho do arquivo: {path}")
            headers[str(path)] = None

    # Verifica se todos os cabeçalhos são iguais
    unique_headers = set(tuple(h) for h in headers.values() if h is not None)
    all_equal = len(unique_headers) == 1

    return headers, all_equal



In [0]:
dataset_paths = [
    dataset_2018_rj,
    dataset_2019_rj,
    dataset_2020_rj,
    dataset_2021_rj,
    dataset_2022_rj,
    dataset_2023_rj,
    dataset_2024_rj,
]
headers, all_equal = check_and_compare_headers(*dataset_paths)
print("Todos os cabeçalhos são iguais?", all_equal)
for path, header in headers.items():
    print(f"{path}: {header}")

Todos os cabeçalhos são iguais? False
data/staged/2018.csv: ['Data', 'HORA (UTC)', 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)', 'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)', 'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)', 'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)', 'RADIACAO GLOBAL (KJ/m²)', 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)', 'TEMPERATURA DO PONTO DE ORVALHO (°C)', 'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)', 'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)', 'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)', 'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)', 'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)', 'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)', 'UMIDADE RELATIVA DO AR, HORARIA (%)', 'VENTO, DIREÇÃO HORARIA (gr) (° (gr))', 'VENTO, RAJADA MAXIMA (m/s)', 'VENTO, VELOCIDADE HORARIA (m/s)', 'REGIAO', 'UF', 'ESTACAO', 'CODIGO (WMO)', 'LATITUDE', 'LONGITUDE', 'ALTITUDE', 'DATA DE FUNDAÇÃO (YYYY-MM-DD)']
data/staged/2019.csv: ['Data', 'Hora UTC', 'PRECIPITAÇÃO TOTAL, H

In [0]:
colunas_padrao = [
    "data","hora_utc","precipitacao_total_horario_mm",
    "pressao_atmosferica_ao_nivel_da_estacao_horaria_mb",
    "pressao_atmosferica_max_na_hora_ant_aut_mb",
    "pressao_atmosferica_min_na_hora_ant_aut_mb",
    "radiacao_global_kj_m2",
    "temperatura_do_ar_bulbo_seco_horaria_degc",
    "temperatura_do_ponto_de_orvalho_degc",
    "temperatura_maxima_na_hora_ant_aut_degc",
    "temperatura_minima_na_hora_ant_aut_degc",
    "temperatura_orvalho_max_na_hora_ant_aut_degc",
    "temperatura_orvalho_min_na_hora_ant_aut_degc",
    "umidade_rel_max_na_hora_ant_aut",
    "umidade_rel_min_na_hora_ant_aut",
    "umidade_relativa_do_ar_horaria",
    "vento_direcao_horaria_gr_deg_gr",
    "vento_rajada_maxima_m_s",
    "vento_velocidade_horaria_m_s",
    "regiao","uf","estacao","codigo_wmo",
    "latitude","longitude","altitude",
    "data_de_fundacao"
]


In [0]:
def substituir_cabecalhos_para_padrao(dataset_paths, colunas_padrao, output_dir=Path("data/staged")):
    """
    Substitui os cabeçalhos dos arquivos CSV em dataset_paths pela lista colunas_padrao, na ordem exata.
    Salva os arquivos resultantes em output_dir com prefixo 'padrao_'.
    """
    for path in dataset_paths:
        try:
            df = pd.read_csv(path)
            # Ajusta o número de colunas se necessário
            if len(df.columns) < len(colunas_padrao):
                # Adiciona colunas extras como NaN
                for col in colunas_padrao[len(df.columns):]:
                    df[col] = pd.NA
            # Renomeia e reordena
            df.columns = colunas_padrao[:len(df.columns)]
            df = df[colunas_padrao]
            out_path = output_dir / f"padrao_{Path(path).name}"
            df.to_csv(out_path, index=False)
            logger.info(f"Cabeçalho substituído para padrão em: {out_path}")
        except Exception as e:
            logger.exception(f"Erro ao processar {path}: {e}")

In [0]:
substituir_cabecalhos_para_padrao(dataset_paths, colunas_padrao)

INFO:__main__:Cabeçalho substituído para padrão em: data/staged/padrao_2018.csv
INFO:__main__:Cabeçalho substituído para padrão em: data/staged/padrao_2019.csv
INFO:__main__:Cabeçalho substituído para padrão em: data/staged/padrao_2020.csv
INFO:__main__:Cabeçalho substituído para padrão em: data/staged/padrao_2021.csv
INFO:__main__:Cabeçalho substituído para padrão em: data/staged/padrao_2022.csv
INFO:__main__:Cabeçalho substituído para padrão em: data/staged/padrao_2023.csv
INFO:__main__:Cabeçalho substituído para padrão em: data/staged/padrao_2024.csv


In [0]:
dataset_2018_rj_harmonized = Path("/Workspace/MVP/data/staged/padrao_2018.csv")
dataset_2019_rj_harmonized = Path("/Workspace/MVP/data/staged/padrao_2019.csv")
dataset_2020_rj_harmonized = Path("/Workspace/MVP/data/staged/padrao_2020.csv")
dataset_2021_rj_harmonized = Path("/Workspace/MVP/data/staged/padrao_2021.csv")
dataset_2022_rj_harmonized = Path("/Workspace/MVP/data/staged/padrao_2022.csv")
dataset_2023_rj_harmonized = Path("/Workspace/MVP/data/staged/padrao_2023.csv")
dataset_2024_rj_harmonized = Path("/Workspace/MVP/data/staged/padrao_2024.csv")

In [0]:

dataset_paths_harmonized = [
    dataset_2018_rj_harmonized,
    dataset_2019_rj_harmonized,
    dataset_2020_rj_harmonized,
    dataset_2021_rj_harmonized,
    dataset_2022_rj_harmonized,
    dataset_2023_rj_harmonized,
    dataset_2024_rj_harmonized,
]
headers, all_equal = check_and_compare_headers(*dataset_paths_harmonized)
print("Todos os cabeçalhos são iguais?", all_equal)
for path, header in headers.items():
    print(f"{path}: {header}")

Todos os cabeçalhos são iguais? True
/Workspace/MVP/data/staged/padrao_2018.csv: ['data', 'hora_utc', 'precipitacao_total_horario_mm', 'pressao_atmosferica_ao_nivel_da_estacao_horaria_mb', 'pressao_atmosferica_max_na_hora_ant_aut_mb', 'pressao_atmosferica_min_na_hora_ant_aut_mb', 'radiacao_global_kj_m2', 'temperatura_do_ar_bulbo_seco_horaria_degc', 'temperatura_do_ponto_de_orvalho_degc', 'temperatura_maxima_na_hora_ant_aut_degc', 'temperatura_minima_na_hora_ant_aut_degc', 'temperatura_orvalho_max_na_hora_ant_aut_degc', 'temperatura_orvalho_min_na_hora_ant_aut_degc', 'umidade_rel_max_na_hora_ant_aut', 'umidade_rel_min_na_hora_ant_aut', 'umidade_relativa_do_ar_horaria', 'vento_direcao_horaria_gr_deg_gr', 'vento_rajada_maxima_m_s', 'vento_velocidade_horaria_m_s', 'regiao', 'uf', 'estacao', 'codigo_wmo', 'latitude', 'longitude', 'altitude', 'data_de_fundacao']
/Workspace/MVP/data/staged/padrao_2019.csv: ['data', 'hora_utc', 'precipitacao_total_horario_mm', 'pressao_atmosferica_ao_nivel_da_

In [0]:

def limpar_e_normalizar_spark(df, colunas_padrao):
    # Remove linhas com nulos nas colunas principais
    df = df.dropna(subset=colunas_padrao)
    
    # Normalização: substitui vírgulas por ponto e converte para float nas colunas numéricas
    colunas_numericas = [
        "precipitacao_total_horario_mm",
        "pressao_atmosferica_ao_nivel_da_estacao_horaria_mb",
        "pressao_atmosferica_max_na_hora_ant_aut_mb",
        "pressao_atmosferica_min_na_hora_ant_aut_mb",
        "radiacao_global_kj_m2",
        "temperatura_do_ar_bulbo_seco_horaria_degc",
        "temperatura_do_ponto_de_orvalho_degc",
        "temperatura_maxima_na_hora_ant_aut_degc",
        "temperatura_minima_na_hora_ant_aut_degc",
        "temperatura_orvalho_max_na_hora_ant_aut_degc",
        "temperatura_orvalho_min_na_hora_ant_aut_degc",
        "umidade_rel_max_na_hora_ant_aut",
        "umidade_rel_min_na_hora_ant_aut",
        "umidade_relativa_do_ar_horaria",
        "vento_direcao_horaria_gr_deg_gr",
        "vento_rajada_maxima_m_s",
        "vento_velocidade_horaria_m_s",
        "latitude",
        "longitude",
        "altitude"
    ]
    for col_name in colunas_numericas:
        if col_name in df.columns:
            df = df.withColumn(col_name, regexp_replace(col(col_name), ",", "."))
            df = df.withColumn(col_name, col(col_name).cast("double"))
    return df

# Exemplo de uso para um dos datasets harmonizados
df_spark = read_dataset_with_spark(dataset_2018_rj_harmonized)
df_spark_limpo = limpar_e_normalizar_spark(df_spark, colunas_padrao)
df_spark_limpo.show(5)

INFO:__main__:Dataset carregado com sucesso: /Workspace/MVP/data/staged/padrao_2018.csv


+----------+--------+-----------------------------+--------------------------------------------------+------------------------------------------+------------------------------------------+---------------------+-----------------------------------------+------------------------------------+---------------------------------------+---------------------------------------+--------------------------------------------+--------------------------------------------+-------------------------------+-------------------------------+------------------------------+-------------------------------+-----------------------+----------------------------+------+---+------------+----------+--------+---------+--------+----------------+
|      data|hora_utc|precipitacao_total_horario_mm|pressao_atmosferica_ao_nivel_da_estacao_horaria_mb|pressao_atmosferica_max_na_hora_ant_aut_mb|pressao_atmosferica_min_na_hora_ant_aut_mb|radiacao_global_kj_m2|temperatura_do_ar_bulbo_seco_horaria_degc|temperatura_do_ponto_de_orva

In [0]:
from pyspark.sql.functions import to_date, to_timestamp, regexp_replace, when, concat, lit

def normalizar_colunas_tempo(df):
    # Normaliza coluna 'data' para tipo DATE, tenta múltiplos formatos
    if 'data' in df.columns:
        df = df.withColumn(
            'data',
            when(
                col('data').rlike(r'^\d{4}/\d{2}/\d{2}$'),
                to_date(col('data'), 'yyyy/MM/dd')
            ).otherwise(
                to_date(col('data'), 'dd/MM/yy')
            )
        )
    # Normaliza coluna 'hora_utc' para formato HH:mm
    if 'hora_utc' in df.columns:
        df = df.withColumn(
            'hora_utc',
            regexp_replace(col('hora_utc'), ' UTC', '')
        )
        df = df.withColumn(
            'hora_utc',
            when(
                col('hora_utc').rlike('^[0-9]{4}$'),
                concat(col('hora_utc').substr(1,2), lit(':'), col('hora_utc').substr(3,2))
            ).otherwise(col('hora_utc'))
        )
    # Normaliza coluna 'data_de_fundacao' para tipo DATE se existir, tenta múltiplos formatos
    if 'data_de_fundacao' in df.columns:
        df = df.withColumn(
            'data_de_fundacao',
            when(
                col('data_de_fundacao').rlike(r'^\d{4}/\d{2}/\d{2}$'),
                to_date(col('data_de_fundacao'), 'yyyy/MM/dd')
            ).otherwise(
                to_date(col('data_de_fundacao'), 'dd/MM/yy')
            )
        )
    return df

# Exemplo de uso:
df_normalizado = normalizar_colunas_tempo(df_spark_limpo)
df_normalizado.show(5)

+----------+--------+-----------------------------+--------------------------------------------------+------------------------------------------+------------------------------------------+---------------------+-----------------------------------------+------------------------------------+---------------------------------------+---------------------------------------+--------------------------------------------+--------------------------------------------+-------------------------------+-------------------------------+------------------------------+-------------------------------+-----------------------+----------------------------+------+---+------------+----------+--------+---------+--------+----------------+
|      data|hora_utc|precipitacao_total_horario_mm|pressao_atmosferica_ao_nivel_da_estacao_horaria_mb|pressao_atmosferica_max_na_hora_ant_aut_mb|pressao_atmosferica_min_na_hora_ant_aut_mb|radiacao_global_kj_m2|temperatura_do_ar_bulbo_seco_horaria_degc|temperatura_do_ponto_de_orva

In [0]:
#PROCESSED_DIR = Path("/Workspace/MVP/data/processed/")
PROCESSED_DIR = Path("/Volumes/workspace/mvp/processed/")
#PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [0]:
# Lê e limpa todos os datasets harmonizados com Spark, junta e salva o resultado
def processar_e_salvar_todos_datasets(dataset_paths, colunas_padrao, output_path):
    dfs_limpos = []
    for path in dataset_paths:
        df = read_dataset_with_spark(path)
        if df is not None:
            df_limpo = limpar_e_normalizar_spark(df, colunas_padrao)
            df_limpo = normalizar_colunas_tempo(df_limpo)
            dfs_limpos.append(df_limpo)
    if dfs_limpos:
        df_final = dfs_limpos[0]
        for df in dfs_limpos[1:]:
            df_final = df_final.unionByName(df)
        df_final.write.mode("overwrite").option("header", True).csv(str(output_path))
        logger.info(f"Dataset final salvo em: {output_path}")
    else:
        logger.warning("Nenhum dataset foi processado.")

In [0]:
processar_e_salvar_todos_datasets(dataset_paths_harmonized, colunas_padrao, PROCESSED_DIR/"weather_data_cleaned_spark.csv")

INFO:__main__:Dataset carregado com sucesso: /Workspace/MVP/data/staged/padrao_2018.csv
INFO:__main__:Dataset carregado com sucesso: /Workspace/MVP/data/staged/padrao_2019.csv
INFO:__main__:Dataset carregado com sucesso: /Workspace/MVP/data/staged/padrao_2020.csv
INFO:__main__:Dataset carregado com sucesso: /Workspace/MVP/data/staged/padrao_2021.csv
INFO:__main__:Dataset carregado com sucesso: /Workspace/MVP/data/staged/padrao_2022.csv
INFO:__main__:Dataset carregado com sucesso: /Workspace/MVP/data/staged/padrao_2023.csv
INFO:__main__:Dataset carregado com sucesso: /Workspace/MVP/data/staged/padrao_2024.csv
INFO:__main__:Dataset final salvo em: /Volumes/workspace/mvp/processed/weather_data_cleaned_spark.csv


In [0]:
import pandas as pd

# Lê todos os arquivos harmonizados e concatena em um único DataFrame pandas
dfs = [pd.read_csv(path) for path in dataset_paths_harmonized]
df_concat = pd.concat(dfs, ignore_index=True)

# Salva em um único arquivo CSV padrão
output_csv_path = PROCESSED_DIR/"weather_data_cleaned_pandas.csv"
df_concat.to_csv(output_csv_path, index=False)
print(f"Arquivo salvo em: {output_csv_path}")

Arquivo salvo em: /Volumes/workspace/mvp/processed/weather_data_cleaned_pandas.csv


In [0]:
import re

def substituir_virgula_por_ponto_em_csv(input_path, output_path):
    """
    Lê um arquivo CSV, substitui vírgulas por ponto nos valores decimais e salva o resultado.
    Exemplo: '3,1' vira '3.1'.
    """
    with open(input_path, 'r', encoding='utf-8') as fin, open(output_path, 'w', encoding='utf-8') as fout:
        for line in fin:
            # Substitui apenas números decimais com vírgula por ponto
            line_mod = re.sub(r'(\d+),(\d+)', r'\1.\2', line)
            fout.write(line_mod)

# Exemplo de uso:
input_csv = output_csv_path  # Use o caminho do arquivo CSV salvo
output_csv = PROCESSED_DIR / "weather_data_cleaned_final.csv"
substituir_virgula_por_ponto_em_csv(input_csv, output_csv)

In [0]:
# Remove aspas de todas as colunas numéricas do DataFrame pandas df_concat

colunas_numericas = [
    "precipitacao_total_horario_mm",
    "pressao_atmosferica_ao_nivel_da_estacao_horaria_mb",
    "pressao_atmosferica_max_na_hora_ant_aut_mb",
    "pressao_atmosferica_min_na_hora_ant_aut_mb",
    "radiacao_global_kj_m2",
    "temperatura_do_ar_bulbo_seco_horaria_degc",
    "temperatura_do_ponto_de_orvalho_degc",
    "temperatura_maxima_na_hora_ant_aut_degc",
    "temperatura_minima_na_hora_ant_aut_degc",
    "temperatura_orvalho_max_na_hora_ant_aut_degc",
    "temperatura_orvalho_min_na_hora_ant_aut_degc",
    "umidade_rel_max_na_hora_ant_aut",
    "umidade_rel_min_na_hora_ant_aut",
    "umidade_relativa_do_ar_horaria",
    "vento_direcao_horaria_gr_deg_gr",
    "vento_rajada_maxima_m_s",
    "vento_velocidade_horaria_m_s",
    "latitude",
    "longitude",
    "altitude"
]

for col in colunas_numericas:
    if col in df_concat.columns:
        df_concat[col] = df_concat[col].astype(str).str.replace('"', '').str.replace("'", '')


df_concat[colunas_numericas].head()

,precipitacao_total_horario_mm,pressao_atmosferica_ao_nivel_da_estacao_horaria_mb,pressao_atmosferica_max_na_hora_ant_aut_mb,pressao_atmosferica_min_na_hora_ant_aut_mb,radiacao_global_kj_m2,temperatura_do_ar_bulbo_seco_horaria_degc,temperatura_do_ponto_de_orvalho_degc,temperatura_maxima_na_hora_ant_aut_degc,temperatura_minima_na_hora_ant_aut_degc,temperatura_orvalho_max_na_hora_ant_aut_degc,temperatura_orvalho_min_na_hora_ant_aut_degc,umidade_rel_max_na_hora_ant_aut,umidade_rel_min_na_hora_ant_aut,umidade_relativa_do_ar_horaria,vento_direcao_horaria_gr_deg_gr,vento_rajada_maxima_m_s,vento_velocidade_horaria_m_s,latitude,longitude,altitude
0,0,"1007,9","1007,9","1007,3",-9999,"25,1","23,2","25,7","25,1","23,4","23,1",89.0,87.0,89.0,291.0,"1,3",",4",-22.0,-42,19
1,0,"1008,1","1008,4","1007,9",-9999,"24,7","23,1","25,1","24,7","23,2",23,91.0,89.0,91.0,311.0,"1,4",",4",-22.0,-42,19
2,0,1008,"1008,3",1008,-9999,24,"22,4","24,7",24,"23,1","22,4",91.0,90.0,91.0,313.0,"1,9",",5",-22.0,-42,19
3,0,"1007,6",1008,"1007,6",-9999,"23,4",22,24,"23,3","22,4",22,92.0,91.0,92.0,327.0,"1,1",",5",-22.0,-42,19
4,0,"1007,1","1007,6","1007,1",-9999,23,"21,8","23,4",23,22,"21,8",93.0,92.0,93.0,224.0,"1,2",",3",-22.0,-42,19


In [0]:
# Checar o tipo de cada coluna do DataFrame pandas df_concat
print(df_concat.dtypes)

data                                                  object
hora_utc                                              object
precipitacao_total_horario_mm                         object
pressao_atmosferica_ao_nivel_da_estacao_horaria_mb    object
pressao_atmosferica_max_na_hora_ant_aut_mb            object
pressao_atmosferica_min_na_hora_ant_aut_mb            object
radiacao_global_kj_m2                                 object
temperatura_do_ar_bulbo_seco_horaria_degc             object
temperatura_do_ponto_de_orvalho_degc                  object
temperatura_maxima_na_hora_ant_aut_degc               object
temperatura_minima_na_hora_ant_aut_degc               object
temperatura_orvalho_max_na_hora_ant_aut_degc          object
temperatura_orvalho_min_na_hora_ant_aut_degc          object
umidade_rel_max_na_hora_ant_aut                       object
umidade_rel_min_na_hora_ant_aut                       object
umidade_relativa_do_ar_horaria                        object
vento_direcao_horaria_gr

In [0]:
# Converte as colunas numéricas do DataFrame pandas df_concat para float
for col in colunas_numericas:
    if col in df_concat.columns:
        df_concat[col] = pd.to_numeric(df_concat[col].str.replace(',', '.'), errors='coerce')

df_concat[colunas_numericas].dtypes

precipitacao_total_horario_mm                         float64
pressao_atmosferica_ao_nivel_da_estacao_horaria_mb    float64
pressao_atmosferica_max_na_hora_ant_aut_mb            float64
pressao_atmosferica_min_na_hora_ant_aut_mb            float64
radiacao_global_kj_m2                                 float64
temperatura_do_ar_bulbo_seco_horaria_degc             float64
temperatura_do_ponto_de_orvalho_degc                  float64
temperatura_maxima_na_hora_ant_aut_degc               float64
temperatura_minima_na_hora_ant_aut_degc               float64
temperatura_orvalho_max_na_hora_ant_aut_degc          float64
temperatura_orvalho_min_na_hora_ant_aut_degc          float64
umidade_rel_max_na_hora_ant_aut                       float64
umidade_rel_min_na_hora_ant_aut                       float64
umidade_relativa_do_ar_horaria                        float64
vento_direcao_horaria_gr_deg_gr                       float64
vento_rajada_maxima_m_s                               float64
vento_ve

In [0]:
# Converte as colunas de data para o formato datetime no DataFrame pandas df_concat

for col in ['data', 'data_de_fundacao']:
    if col in df_concat.columns:
        df_concat[col] = pd.to_datetime(df_concat[col], errors='coerce', infer_datetime_format=True)

df_concat[['data', 'data_de_fundacao']].head()

/home/spark-675c860a-6b7f-4b72-b5a4-f9/.ipykernel/81/command-7651223616587102-1985679400:5: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_concat[col] = pd.to_datetime(df_concat[col], errors='coerce', infer_datetime_format=True)
/home/spark-675c860a-6b7f-4b72-b5a4-f9/.ipykernel/81/command-7651223616587102-1985679400:5: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_concat[col] = pd.to_datetime(df_concat[col], errors='coerce', infer_datetime_format=True)


,data,data_de_fundacao
0,2018-01-01,2015-08-28
1,2018-01-01,2015-08-28
2,2018-01-01,2015-08-28
3,2018-01-01,2015-08-28
4,2018-01-01,2015-08-28


In [0]:
# Checar a tipagem de todas as colunas do DataFrame pandas df_concat
print(df_concat.dtypes)

data                                                  datetime64[ns]
hora_utc                                                      object
precipitacao_total_horario_mm                                float64
pressao_atmosferica_ao_nivel_da_estacao_horaria_mb           float64
pressao_atmosferica_max_na_hora_ant_aut_mb                   float64
pressao_atmosferica_min_na_hora_ant_aut_mb                   float64
radiacao_global_kj_m2                                        float64
temperatura_do_ar_bulbo_seco_horaria_degc                    float64
temperatura_do_ponto_de_orvalho_degc                         float64
temperatura_maxima_na_hora_ant_aut_degc                      float64
temperatura_minima_na_hora_ant_aut_degc                      float64
temperatura_orvalho_max_na_hora_ant_aut_degc                 float64
temperatura_orvalho_min_na_hora_ant_aut_degc                 float64
umidade_rel_max_na_hora_ant_aut                              float64
umidade_rel_min_na_hora_ant_aut   

In [0]:
# Salva o DataFrame pandas df_concat em um novo arquivo CSV
novo_output_csv = PROCESSED_DIR / "weather_data_cleaned_final_v2.csv"
df_concat.to_csv(novo_output_csv, index=False)
print(f"Arquivo salvo em: {novo_output_csv}")

Arquivo salvo em: /Volumes/workspace/mvp/processed/weather_data_cleaned_final_v2.csv
